# 09 — Adaptive Ensemble + Submission (Upgrades 2 + 4)

Notebook cuối cùng — chứa **Upgrade 2 (4-round iterative)** + **Upgrade 4 (adaptive weighting)** + submission generation.

## Phương pháp

**Adaptive weighting (Confidence-Calibrated F):** per-query, per-model 4 confidence signals → softmax weights với floor η=0.05.

```
s̃_m(q,g) = (s_m(q,g) − μ_m(q)) / (σ_m(q) + ε)      # z-norm per query

c_m(q) = α·top1_norm + β·margin − γ·H + λ·agreement
w_m(q) = (1 − M·η)·softmax(c_m(q) / T) + η
S(q,g) = Σ_m w_m(q) · s̃_m(q,g)                     # single-pass fusion
```

**Alternative iterative 4-round (Upgrade 2, ablation):**
```
S^(0) = s̃_UIT
S^(1) = w_2 · S^(0) + (1−w_2) · s̃_BLIP2     # w_2 = 0.9125 paper-fixed
S^(2) = w_3 · S^(1) + (1−w_3) · s̃_CLIP       # w_3 = 0.925  paper-fixed
S^(3) = w_4 · S^(2) + (1−w_4) · s̃_PE_G14     # w_4 = 0.85   new
```

**Validation gate:** chỉ enable adaptive nếu beat fixed-w trên **cả** val_split + val_zeroshot_scene.

**Inputs:**
- `features/{uit,blip2,clip,pe_g14}/scores_*_rr.pt` (k-reciprocal reranked, từ 08)
- `manifests/gallery_manifest.parquet`, `query_manifest.parquet`
- val score files cho gate

**Outputs:**
- `submission/answer.txt`, `submission/answer.zip`
- `validation/adaptive_weights_meta.json` (winner decision)
- `validation/ablation_table.parquet`
- `submission/debug_topk.parquet`

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, warnings
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from aic_colab_utils import (
    setup_aic2026_environment, select_a100_device,
    sync_chunk_to_drive, wait_for_pending_syncs,
)

PATHS = setup_aic2026_environment()
MANIFEST_DIR = PATHS['manifest_dir']
LOCAL_ROOT = PATHS['local_root']; OUTPUT_ROOT = PATHS['output_root']
DRIVE_OUTPUT_ROOT = PATHS['drive_output_root']
FEAT_ROOT = OUTPUT_ROOT / 'features'
SUB_DIR = OUTPUT_ROOT / 'submission'
VAL_DIR = OUTPUT_ROOT / 'validation'
SUB_DIR.mkdir(parents=True, exist_ok=True)
VAL_DIR.mkdir(parents=True, exist_ok=True)
device = select_a100_device()

In [ ]:
# === Load 4 score matrices (prefer reranked _rr versions) ===
# Order: UIT (Round 1) → BLIP-2 (R2) → CLIP (R3) → PE-G14 (R4)
MODEL_SLOT = [
    ('uit',   'scores_uit'),
    ('blip2', 'scores_blip2'),
    ('clip',  'scores_clip'),
    ('pe_g14','scores_pe'),
]

def _load(model, base):
    p_rr = FEAT_ROOT / model / f'{base}_rr.pt'
    p = FEAT_ROOT / model / f'{base}.pt'
    chosen = p_rr if p_rr.exists() else p
    if not chosen.exists():
        raise FileNotFoundError(f'Missing scores for {model}: tried {p_rr}, {p}')
    return torch.load(chosen, map_location='cpu')

score_payloads = [_load(m, base) for m, base in MODEL_SLOT]
for (m, _), pay in zip(MODEL_SLOT, score_payloads):
    print(f'{m:10s} → {tuple(pay["scores"].shape)}')

# Align query/gallery IDs across all models (use UIT's order as canonical)
canon_q_ids = np.asarray([str(x) for x in score_payloads[0]['query_ids']])
canon_g_ids = np.asarray([str(x) for x in score_payloads[0]['gallery_ids']])
Q, G_ = len(canon_q_ids), len(canon_g_ids)
print(f'canonical: Q={Q} G={G_}')

def _align(pay):
    qi = {str(x): i for i, x in enumerate(pay['query_ids'])}
    gi = {str(x): i for i, x in enumerate(pay['gallery_ids'])}
    q_idx = np.array([qi[q] for q in canon_q_ids], dtype=np.int64)
    g_idx = np.array([gi[g] for g in canon_g_ids], dtype=np.int64)
    S = pay['scores'].float()
    return S[q_idx][:, g_idx]

S_list = [_align(p) for p in score_payloads]
for m, (n, _) in zip(MODEL_SLOT, MODEL_SLOT):
    pass
print('All score matrices aligned.')

In [ ]:
# === Per-query z-normalization per model ===
# Critical: BLIP-2 ITM logits vs cosine of others → very different scales.
# Without z-norm one model dominates.

def per_query_zscore(S):
    mu = S.mean(dim=1, keepdim=True)
    sd = S.std(dim=1, keepdim=True) + 1e-6
    return (S - mu) / sd

Stil_list = [per_query_zscore(S.to(device)) for S in S_list]
print('z-normalized per query.')

In [ ]:
# === Compute 4 per-(model, query) confidence signals ===
K_TOPK = 20
TAU_E = 0.05  # entropy softmax temperature

def compute_confidence(S_list, k=K_TOPK):
    """Returns:
        c[M, Q, 4]  raw confidence components (top1_norm, margin, neg_entropy, agreement)
        topk_idx[M, Q, k]  top-k gallery indices per query (for agreement)
    """
    M = len(S_list)
    Q_, G__ = S_list[0].shape
    out = torch.zeros(M, Q_, 4, device=device)
    topk_idx = []
    topk_vals = []
    for m, S in enumerate(S_list):
        vals, idx = torch.topk(S, k, dim=1)  # vals desc-sorted
        smin = S.min(dim=1, keepdim=True).values
        smax = S.max(dim=1, keepdim=True).values
        top1 = vals[:, 0:1]
        top1_norm = ((top1 - smin) / (smax - smin + 1e-6)).squeeze(1)
        rest = vals[:, 1:]
        margin = (vals[:, 0] - rest.mean(dim=1)) / (rest.std(dim=1) + 1e-6)
        # Entropy over softmax(top-K / tau_e)
        lg = vals / TAU_E
        lg = lg - lg.max(dim=1, keepdim=True).values
        p = torch.softmax(lg, dim=1)
        H = -(p * (p.clamp_min(1e-12)).log()).sum(dim=1) / np.log(k)
        out[m, :, 0] = top1_norm
        out[m, :, 1] = margin
        out[m, :, 2] = -H  # neg-entropy: higher = more confident
        topk_idx.append(idx)
        topk_vals.append(vals)
    # Agreement: Spearman-on-union proxy via fraction of top-k common across models
    for m in range(M):
        for q in range(Q_):
            my = set(topk_idx[m][q].tolist())
            others = []
            for mp in range(M):
                if mp == m: continue
                inter = len(my & set(topk_idx[mp][q].tolist()))
                others.append(inter / k)
            out[m, q, 3] = sum(others) / max(1, len(others))
    return out, topk_idx, topk_vals

conf, topk_idx, topk_vals = compute_confidence(Stil_list)
print('Confidence tensor:', tuple(conf.shape))

In [ ]:
# === Adaptive weights (Confidence-Calibrated F) ===
ALPHA = 1.0    # top1_norm weight
BETA = 1.0     # margin weight
GAMMA = 0.5    # entropy weight (already negated)
LAM = 0.5      # agreement weight
TEMP_W = 1.0   # softmax temperature
ETA = 0.05     # weight floor

def adaptive_weights(conf, alpha=ALPHA, beta=BETA, gamma=GAMMA, lam=LAM, T=TEMP_W, eta=ETA):
    # conf[m, q, :] = [top1_norm, margin, -H_norm, agreement]
    c = alpha * conf[..., 0] + beta * conf[..., 1] + gamma * conf[..., 2] + lam * conf[..., 3]
    # c shape (M, Q)
    cT = c / T
    cT = cT - cT.max(dim=0, keepdim=True).values
    w = torch.softmax(cT, dim=0)
    M = w.shape[0]
    w = (1 - M * eta) * w + eta
    return w  # (M, Q), sums to 1 per Q

W = adaptive_weights(conf)
print('Adaptive weights shape:', tuple(W.shape), 'sum-per-q sample:', float(W[:, 0].sum()))
for i, (m, _) in enumerate(MODEL_SLOT):
    print(f'  {m:10s}: mean_w={float(W[i].mean()):.4f}  std_w={float(W[i].std()):.4f}')

In [ ]:
# === Single-pass adaptive fusion (Upgrade 4) ===
S_adaptive = torch.zeros_like(Stil_list[0])
for m in range(len(Stil_list)):
    S_adaptive += W[m, :, None] * Stil_list[m]

# === Iterative 4-round (Upgrade 2) ===
#
# CRITICAL FIX: paper's iterative form đặt base model làm S^(0) và "kéo nhẹ"
# các model sau bằng (1-w)·new. Effective contribution sau 4 round:
#   base · w_2 · w_3 · w_4  (lớn nhất)
#
# Với MODEL_SLOT cũ (UIT→BLIP2→CLIP→PE_G14) và w=[0.9125, 0.925, 0.85]:
#   UIT     = 0.9125·0.925·0.85 = 0.717  (71.7%)
#   BLIP-2  = 0.0875·0.925·0.85 = 0.069
#   CLIP    = 0.075·0.85         = 0.064
#   PE-G14  = 0.15               = 0.150
# → UIT chiếm 71.7% mặc dù PE-G14 mạnh hơn standalone. Sai.
#
# FIX: đảo ngược thứ tự — strongest first, weakest last. PE-G14 làm base.
# Order: PE-G14 (R1) → CLIP (R2) → BLIP-2 (R3) → UIT (R4)
# Weights paper giữ nguyên ý nghĩa: mỗi round kéo nhẹ ~7-15%.
#   PE-G14  = 0.85·0.925·0.9125 = 0.717  (71.7%)  ← dominant, đúng intent
#   CLIP    = 0.15·0.925·0.9125 = 0.127
#   BLIP-2  = 0.075·0.9125      = 0.068
#   UIT     = 0.0875            = 0.088

# Map slot name → index in Stil_list (theo MODEL_SLOT order: uit=0, blip2=1, clip=2, pe_g14=3)
SLOT_IDX = {name: i for i, (name, _) in enumerate(MODEL_SLOT)}

# Reversed iterative order: strongest standalone → weakest
ITER_ORDER = ['pe_g14', 'clip', 'blip2', 'uit']      # R1=PE-G14, R4=UIT
ITER_WEIGHTS = [0.85, 0.925, 0.9125]                  # mix-in weights for R2, R3, R4

S_iter = Stil_list[SLOT_IDX[ITER_ORDER[0]]].clone()   # base = PE-G14
for t, (model_name, w_t) in enumerate(zip(ITER_ORDER[1:], ITER_WEIGHTS), start=1):
    new_S = Stil_list[SLOT_IDX[model_name]]
    S_iter = w_t * S_iter + (1 - w_t) * new_S

# === Fixed-weight single-pass baseline (4-way uniform mean) ===
S_uniform = sum(Stil_list) / len(Stil_list)

# === Print effective contributions for transparency ===
def _effective_contributions(order, weights):
    contrib = {order[0]: 1.0}
    # Each round: existing_models *= w; new_model = 1 - w
    for t, m in enumerate(order[1:]):
        w_t = weights[t]
        for k in contrib:
            contrib[k] *= w_t
        contrib[m] = 1 - w_t
    return contrib

contrib_iter = _effective_contributions(ITER_ORDER, ITER_WEIGHTS)
print('Iterative effective contributions:')
for m, c in contrib_iter.items():
    print(f'  {m:10s}: {c*100:.1f}%')
print('Fusion variants ready: S_adaptive, S_iter (4-round, PE-G14 base), S_uniform')


In [ ]:
# === Validation harness: mAP@10 + R@1/5/10 trên val + val_zs ===
#
# CRITICAL: AIC 2026 Track 4 chấm điểm bằng mAP, NOT recall. mAP@10 sát với
# leaderboard hơn (submission top-10) so với R@1 alone.
# Single-positive setting: AP = 1/(rank+1) if rank<K else 0.

def _load_val_scores(slot_name, slot_base):
    out = {}
    for split in ('val', 'val_zs'):
        p_rr = FEAT_ROOT / slot_name / f'scores_{split}_rr.pt'
        p = FEAT_ROOT / slot_name / f'scores_{split}.pt'
        chosen = p_rr if p_rr.exists() else (p if p.exists() else None)
        if chosen is None:
            out[split] = None
        else:
            pay = torch.load(chosen, map_location='cpu')
            out[split] = pay
    return out

val_payloads = {m: _load_val_scores(m, b) for m, b in MODEL_SLOT}

def _build_val_S_list(split):
    payloads = [val_payloads[m][split] for m, _ in MODEL_SLOT]
    if any(p is None for p in payloads):
        return None, None, None
    canon_q = np.asarray([str(x) for x in payloads[0]['query_ids']])
    canon_g = np.asarray([str(x) for x in payloads[0]['gallery_ids']])
    S_list = []
    for p in payloads:
        qi = {str(x): i for i, x in enumerate(p['query_ids'])}
        gi = {str(x): i for i, x in enumerate(p['gallery_ids'])}
        q_idx = np.array([qi[q] for q in canon_q], dtype=np.int64)
        g_idx = np.array([gi[g] for g in canon_g], dtype=np.int64)
        S_list.append(p['scores'].float()[q_idx][:, g_idx])
    return [per_query_zscore(S.to(device)) for S in S_list], canon_q, canon_g

def _metrics(S, q_ids, g_ids, ks=(1, 5, 10), map_ks=(10, 100)):
    """Single-positive retrieval metrics.

    Returns dict with:
      mAP        — full mAP (rank up to G, AP = 1/(rank+1))
      mAP@K      — capped: AP = 1/(rank+1) if rank<K else 0  ← AIC submission style
      R@K        — recall at K (rank<K)
    """
    n = S.shape[0]
    g_idx_by_id = {str(g): i for i, g in enumerate(g_ids)}
    # Single positive per query: q_id == g_id (val constructed from train rows)
    pos = np.array([g_idx_by_id[str(q)] for q in q_ids], dtype=np.int64)
    Top = torch.argsort(S, dim=1, descending=True).cpu().numpy()
    # Rank of GT per query
    ranks = np.full(n, fill_value=S.shape[1], dtype=np.int64)
    for i in range(n):
        loc = np.where(Top[i] == pos[i])[0]
        if len(loc):
            ranks[i] = loc[0]
    metrics = {}
    metrics['mAP'] = float(np.mean(1.0 / (ranks + 1.0)))
    for k in map_ks:
        ap_k = np.where(ranks < k, 1.0 / (ranks + 1.0), 0.0)
        metrics[f'mAP@{k}'] = float(ap_k.mean())
    for k in ks:
        metrics[f'R@{k}'] = float((ranks < k).mean())
    return metrics

def _eval(split, fusion_fn_name, iter_order=None, iter_weights=None):
    S_v_list, q_ids, g_ids = _build_val_S_list(split)
    if S_v_list is None:
        return None
    conf_v, _, _ = compute_confidence(S_v_list)
    if fusion_fn_name == 'adaptive':
        Wv = adaptive_weights(conf_v)
        Sv = torch.zeros_like(S_v_list[0])
        for m in range(len(S_v_list)):
            Sv += Wv[m, :, None] * S_v_list[m]
    elif fusion_fn_name == 'iter4':
        order = iter_order or ITER_ORDER
        weights = iter_weights or ITER_WEIGHTS
        Sv = S_v_list[SLOT_IDX[order[0]]].clone()
        for t, m_name in enumerate(order[1:]):
            w_t = weights[t]
            Sv = w_t * Sv + (1 - w_t) * S_v_list[SLOT_IDX[m_name]]
    else:  # uniform
        Sv = sum(S_v_list) / len(S_v_list)
    return _metrics(Sv, q_ids, g_ids)

# === Per-model standalone for diagnostic ===
print('=== Per-model standalone (val) ===')
for slot_idx, (m, _) in enumerate(MODEL_SLOT):
    S_v_list, q_ids, g_ids = _build_val_S_list('val')
    if S_v_list is None: continue
    r = _metrics(S_v_list[slot_idx], q_ids, g_ids)
    print(f'  {m:10s}  mAP@10={r["mAP@10"]:.4f}  R@1={r["R@1"]:.4f}  R@10={r["R@10"]:.4f}')

# === Fusion variants ===
print('\n=== Fusion variants (val + val_zs) ===')
ablation = []
configs = [
    ('uniform_4way',   'uniform',  None, None),
    ('iter4_reversed', 'iter4',    ITER_ORDER, ITER_WEIGHTS),  # PE-G14 base (new)
    ('iter4_paper',    'iter4',    ['uit','blip2','clip','pe_g14'], [0.9125, 0.925, 0.85]),  # original (UIT base) for ablation
    ('adaptive_F',     'adaptive', None, None),
]
for cfg_name, cfg_fn, order, weights in configs:
    for split in ('val', 'val_zs'):
        r = _eval(split, cfg_fn, order, weights)
        if r is None:
            print(f'  {cfg_name}/{split}: skipped (missing val scores)')
            continue
        r['config'] = cfg_name; r['split'] = split
        ablation.append(r)
        print(f'  {cfg_name:18s}/{split:8s}  mAP@10={r["mAP@10"]:.4f}  R@1={r["R@1"]:.4f}  R@10={r["R@10"]:.4f}')

# === Grid search iterative weights (find best for PE-G14-base iteration) ===
print('\n=== Grid search ITER_WEIGHTS (PE-G14 base, optimize val mAP@10) ===')
RUN_ITER_GRID = True
best_iter_weights = ITER_WEIGHTS
best_iter_map10 = -1.0
if RUN_ITER_GRID:
    import itertools
    grid_results = []
    for w2, w3, w4 in itertools.product(
        [0.70, 0.80, 0.85, 0.90, 0.95],   # mix-in CLIP
        [0.85, 0.90, 0.925, 0.95],         # mix-in BLIP2
        [0.85, 0.90, 0.9125, 0.95],        # mix-in UIT
    ):
        weights_try = [w2, w3, w4]
        r_val = _eval('val', 'iter4', ITER_ORDER, weights_try)
        r_zs = _eval('val_zs', 'iter4', ITER_ORDER, weights_try)
        if r_val is None: continue
        # Optimize on mean(val, val_zs) mAP@10 (val_zs proxy cho OOD)
        if r_zs is not None:
            score = 0.5 * (r_val['mAP@10'] + r_zs['mAP@10'])
        else:
            score = r_val['mAP@10']
        grid_results.append({
            'w2': w2, 'w3': w3, 'w4': w4,
            'val_mAP@10': r_val['mAP@10'],
            'val_zs_mAP@10': r_zs['mAP@10'] if r_zs else None,
            'score': score,
        })
        if score > best_iter_map10:
            best_iter_map10 = score
            best_iter_weights = weights_try
    grid_df = pd.DataFrame(grid_results).sort_values('score', ascending=False)
    print(f'Best iterative weights: w_2={best_iter_weights[0]}, w_3={best_iter_weights[1]}, w_4={best_iter_weights[2]} → score={best_iter_map10:.4f}')
    print('Top 5 weight configs:')
    print(grid_df.head(5).to_string(index=False))
    grid_df.to_parquet(VAL_DIR / 'iter_weight_grid.parquet', index=False)

    # Re-eval iter4 with best weights and add to ablation
    for split in ('val', 'val_zs'):
        r = _eval(split, 'iter4', ITER_ORDER, best_iter_weights)
        if r is None: continue
        r['config'] = 'iter4_grid_best'; r['split'] = split
        ablation.append(r)
        print(f'  iter4_grid_best /{split:8s}  mAP@10={r["mAP@10"]:.4f}  R@1={r["R@1"]:.4f}')

ablation_df = pd.DataFrame(ablation)
ablation_df.to_parquet(VAL_DIR / 'ablation_table.parquet', index=False)

# Update S_iter to use grid-best weights (sẽ dùng cho submission nếu iter4 wins)
S_iter_grid = Stil_list[SLOT_IDX[ITER_ORDER[0]]].clone()
for t, m_name in enumerate(ITER_ORDER[1:]):
    w_t = best_iter_weights[t]
    S_iter_grid = w_t * S_iter_grid + (1 - w_t) * Stil_list[SLOT_IDX[m_name]]


In [ ]:
# === Auto-fallback gate ===
#
# Pick winner across:
#   uniform_4way     — 4-way uniform mean (sanity baseline)
#   iter4_paper      — paper order (UIT base) — diagnostic ablation
#   iter4_reversed   — PE-G14 base, paper weights (Upgrade 2 fixed)
#   iter4_grid_best  — PE-G14 base, val-tuned weights
#   adaptive_F       — single-pass adaptive (Upgrade 4)
#
# Selection metric: mAP@10 (AIC leaderboard style).
# Must beat baseline on BOTH val + val_zs (zero-shot scene) để enable.

FALLBACK_THRESHOLD = 0.005  # 0.5% mAP@10

def _avg(metric, cfg):
    sub = ablation_df[ablation_df['config'] == cfg]
    if not len(sub): return float('-inf')
    return float(sub[metric].mean())

scores = {
    cfg: _avg('mAP@10', cfg)
    for cfg in ['uniform_4way', 'iter4_paper', 'iter4_reversed', 'iter4_grid_best', 'adaptive_F']
}
print('=== Config mAP@10 (mean of val + val_zs) ===')
for cfg, s in sorted(scores.items(), key=lambda x: -x[1]):
    marker = ' ← winner' if s == max(scores.values()) else ''
    print(f'  {cfg:18s}: {s:.4f}{marker}')

# Default baseline: iter4_reversed (PE-G14 base, paper weights)
baseline_score = scores.get('iter4_reversed', float('-inf'))
chosen = 'iter4_reversed'
chosen_score = baseline_score

# Promote to grid-best if beats baseline by threshold
if scores.get('iter4_grid_best', -1) > baseline_score + FALLBACK_THRESHOLD:
    chosen = 'iter4_grid_best'
    chosen_score = scores['iter4_grid_best']

# Promote to adaptive_F if it beats current chosen
if scores.get('adaptive_F', -1) > chosen_score + FALLBACK_THRESHOLD:
    chosen = 'adaptive_F'
    chosen_score = scores['adaptive_F']

# Last-resort safety: if uniform_4way beats everything (weird), use it
if scores.get('uniform_4way', -1) > chosen_score + FALLBACK_THRESHOLD:
    chosen = 'uniform_4way'
    chosen_score = scores['uniform_4way']

meta = {
    'chosen': chosen,
    'chosen_mAP@10': chosen_score,
    'all_scores': scores,
    'fallback_threshold': FALLBACK_THRESHOLD,
    'metric': 'mAP@10 (single-positive, AP=1/(rank+1) if rank<10 else 0)',
    'adaptive_hyperparams': {
        'alpha': ALPHA, 'beta': BETA, 'gamma': GAMMA, 'lam': LAM,
        'temp': TEMP_W, 'eta': ETA, 'k_topk': K_TOPK, 'tau_e': TAU_E,
    },
    'iter4_reversed': {'order': ITER_ORDER, 'weights': ITER_WEIGHTS},
    'iter4_grid_best': {'order': ITER_ORDER, 'weights': best_iter_weights},
    'iter4_effective_contribution_pct': {k: round(v*100, 2) for k, v in contrib_iter.items()},
    'model_order_slots': [m for m, _ in MODEL_SLOT],
}
with open(VAL_DIR / 'adaptive_weights_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f'\n>>> CHOSEN FUSION: {chosen}  (mAP@10 = {chosen_score:.4f})')


In [ ]:
# === Generate top-10 submission ===
if chosen == 'adaptive_F':
    S_FINAL = S_adaptive
elif chosen == 'iter4_grid_best':
    S_FINAL = S_iter_grid
elif chosen == 'iter4_reversed':
    S_FINAL = S_iter
elif chosen == 'iter4_paper':
    # Recompute paper-style for safety
    S_FINAL = Stil_list[SLOT_IDX['uit']].clone()
    for t, m_name in enumerate(['blip2', 'clip', 'pe_g14']):
        w_t = [0.9125, 0.925, 0.85][t]
        S_FINAL = w_t * S_FINAL + (1 - w_t) * Stil_list[SLOT_IDX[m_name]]
else:
    S_FINAL = S_uniform

TOPK_SUBMIT = 10
top_vals, top_idx = torch.topk(S_FINAL, TOPK_SUBMIT, dim=1)
top_idx_np = top_idx.cpu().numpy()

gallery_df = pd.read_parquet(MANIFEST_DIR / 'gallery_manifest.parquet')
query_df = pd.read_parquet(MANIFEST_DIR / 'query_manifest.parquet')

assert len(canon_g_ids) == len(gallery_df), f'canon_g_ids {len(canon_g_ids)} vs manifest {len(gallery_df)}'
canon_to_manifest_id = canon_g_ids  # already gallery_id strings

lines = []
for q in range(len(canon_q_ids)):
    ids = [canon_to_manifest_id[idx] for idx in top_idx_np[q]]
    assert len(set(ids)) == TOPK_SUBMIT, f'duplicate in query {q}'
    lines.append(' '.join(ids))

assert len(lines) == 1978, f'got {len(lines)} lines, expected 1978'
ans_txt = SUB_DIR / 'answer.txt'
ans_zip = SUB_DIR / 'answer.zip'
ans_txt.write_text('\n'.join(lines) + '\n', encoding='utf-8')
with zipfile.ZipFile(ans_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(ans_txt, arcname='answer.txt')
print(f'Wrote {ans_txt} ({len(lines)} lines)')
print(f'Wrote {ans_zip}')

# Debug topk dump
debug_rows = []
for q in range(len(canon_q_ids)):
    for rank, gidx in enumerate(top_idx_np[q]):
        debug_rows.append({
            'query_id': canon_q_ids[q],
            'rank': rank + 1,
            'gallery_id': canon_to_manifest_id[gidx],
            'score_final': float(top_vals[q, rank]),
            'w_uit': float(W[SLOT_IDX['uit'], q]),
            'w_blip2': float(W[SLOT_IDX['blip2'], q]),
            'w_clip': float(W[SLOT_IDX['clip'], q]),
            'w_pe_g14': float(W[SLOT_IDX['pe_g14'], q]),
            'chosen_fusion': chosen,
        })
debug_df = pd.DataFrame(debug_rows)
debug_df.to_parquet(SUB_DIR / 'debug_topk.parquet', index=False)
print('debug_topk.parquet rows:', len(debug_df))

for f in [ans_txt, ans_zip, SUB_DIR / 'debug_topk.parquet',
          VAL_DIR / 'adaptive_weights_meta.json',
          VAL_DIR / 'ablation_table.parquet',
          VAL_DIR / 'iter_weight_grid.parquet']:
    if f.exists():
        sync_chunk_to_drive(f, LOCAL_ROOT, DRIVE_OUTPUT_ROOT, background=True)
wait_for_pending_syncs()
print(f'Submission ready. Fusion = {chosen}, mAP@10 (val) = {chosen_score:.4f}')
